In [ ]:
print("Wazzup")

In [ ]:
print("Jeg var her -Simon")

In [2]:
this_is_me = 22
print(this_is_me)

22


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft
from scipy.io import wavfile

# =====================================================
# Parameters
# =====================================================

fs = 44100           # Sampling frequency
N = 2000              # FIR filter order

# Gains in dB
gains_db = [0, -5, -10, -15, -20]

# Convert to linear scale
gains = 10**(np.array(gains_db)/20)

# =====================================================
# Design 5-band equalizer
# =====================================================

nyq = fs/2

# 5 equal-width bands
edges = np.array([0,
                  nyq/5,
                  2*nyq/5,
                  3*nyq/5,
                  4*nyq/5,
                  nyq])

# firwin2 requires normalized freq between 0 and 1
freq = np.array([
    edges[0],
    edges[1],
    edges[1],
    edges[2],
    edges[2],
    edges[3],
    edges[3],
    edges[4],
    edges[4],
    edges[5]
]) / nyq

amp = np.array([
    gains[0], gains[0],
    gains[1], gains[1],
    gains[2], gains[2],
    gains[3], gains[3],
    gains[4], gains[4]
])

# FIR filter coefficients
h = signal.firwin2(N + 1, freq, amp)

# =====================================================
# Frequency response
# =====================================================

w, H = signal.freqz(h, worN=4096, fs=fs)

plt.figure(figsize=(8,4))
plt.plot(w, 20*np.log10(np.abs(H)+1e-12))
plt.grid()
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude [dB]")
plt.title("5-Band FIR Equalizer")
plt.xlim([0, fs/2])
plt.show()

# =====================================================
# Zero padding ×2
# =====================================================

h2 = np.concatenate([h, np.zeros(len(h))])

H2 = fft(h2)

f2 = np.arange(len(H2))*fs/len(H2)

plt.figure(figsize=(8,4))
plt.plot(f2[:len(H2)//2],20*np.log10(np.abs(H2[:len(H2)//2])+1e-12))
plt.grid()
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude [dB]")
plt.title("Zero-padding 2x")
plt.show()

# =====================================================
# Zero padding ×10
# =====================================================

h10 = np.concatenate([h, np.zeros(9*len(h))])

H10 = fft(h10)

f10 = np.arange(len(H10))*fs/len(H10)

plt.figure(figsize=(8,4))
plt.plot(f10[:len(H10)//2],
         20*np.log10(np.abs(H10[:len(H10)//2])+1e-12))
plt.grid()
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude [dB]")
plt.title("Zero-padding 10x")
plt.show()

# =====================================================
# White noise test
# =====================================================

x = np.random.randn(50000)

y = signal.lfilter(h, 1, x)

Nfft = 8192

X = fft(x, Nfft)
Y = fft(y, Nfft)

f = np.arange(Nfft)*fs/Nfft

plt.figure(figsize=(8,4))
plt.plot(f[:Nfft//2],
         20*np.log10(np.abs(X[:Nfft//2])+1e-12),
         label="Original noise")

plt.plot(f[:Nfft//2],
         20*np.log10(np.abs(Y[:Nfft//2])+1e-12),
         label="Filtered noise")

plt.grid()
plt.legend()
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude [dB]")
plt.title("White Noise Spectrum")
plt.show()

# =====================================================
# Music file example
# =====================================================

# Uncomment and insert your file name

"""
fs_audio, music = wavfile.read("music.wav")

music = music.astype(float)

filtered_music = signal.lfilter(h, 1, music)

filtered_music = filtered_music / np.max(np.abs(filtered_music))

wavfile.write(
    "equalized_music.wav",
    fs_audio,
    (filtered_music * 32767).astype(np.int16)
)
"""